# 25_nsmc_finetune.ipynb

**12주차 · 3교시** 실습 노트북

- 이론 설명과 관찰 포인트는 배포 자료(`12week/student/`)를 함께 보세요.
- 실행 환경: `%DL2026_HOME%\venv` 활성화 후 `Python (dl2026)` 커널.
- 전체 9셀. 위에서부터 순서대로 실행합니다.

## 1-1. 준비 — 1교시 데이터 재사용

**셀 1** — 1교시와 동일하게 다시 만든다

In [ ]:
import torch, numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import load_dataset

MODEL, MAX_LEN, N_TRAIN, N_TEST = "klue/bert-base", 64, 20000, 4000   # ★ 전원 동일
tok = AutoTokenizer.from_pretrained(MODEL)

raw   = load_dataset("nsmc")
train = raw["train"].shuffle(seed=42).select(range(N_TRAIN))
test  = raw["test"].shuffle(seed=42).select(range(N_TEST))

def tok_fn(b): return tok(b["document"], truncation=True, max_length=MAX_LEN)
train_tok = train.map(tok_fn, batched=True)
test_tok  = test.map(tok_fn,  batched=True)

print(train_tok, "\nGPU :", torch.cuda.is_available())

**셀 2** — 모델 + 지표 함수

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2)
# ↑ "newly initialized" 경고는 정상 — 분류 헤드를 새로 만든 것 (2교시)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = (preds == labels).mean()
    tp = ((preds == 1) & (labels == 1)).sum()
    fp = ((preds == 1) & (labels == 0)).sum()
    fn = ((preds == 0) & (labels == 1)).sum()
    prec = tp / (tp + fp + 1e-9); rec = tp / (tp + fn + 1e-9)
    return {"accuracy": acc, "precision": prec, "recall": rec,
            "f1": 2 * prec * rec / (prec + rec + 1e-9)}          # 6주차 지표 ★

## 1-2. 학습 (실행 6~12분)

**셀 3** — 하이퍼파라미터 ★ 8GB GPU 기준

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding

args = TrainingArguments(
    output_dir="results/nsmc",
    num_train_epochs=1,                  # ★ 시간 제약. 2 로 올리면 +1~2%p
    per_device_train_batch_size=32,      # ★ OOM 이면 16 또는 8
    per_device_eval_batch_size=64,
    learning_rate=2e-5,                  # ★ 파인튜닝은 작은 lr (9주차와 같은 이유)
    fp16=torch.cuda.is_available(),      # ★ 7주차 AMP. VRAM 절반, 속도 2배
    eval_strategy="epoch",               # ⚠️ 구버전은 evaluation_strategy
    logging_steps=50,
    save_strategy="no",                  # 수업 중에는 체크포인트 저장 생략
    seed=42,                             # 6주차 재현성 ★
    report_to="none",
)

trainer = Trainer(
    model=model, args=args,
    train_dataset=train_tok, eval_dataset=test_tok,
    data_collator=DataCollatorWithPadding(tok),   # ★ 배치 안에서만 패딩 → 빠르다
    compute_metrics=compute_metrics,
)
trainer.train()

> **핵심 ★ (출제 지점)**: `learning_rate=2e-5` 는 **처음부터 학습할 때보다 훨씬 작습니다.** 사전학습으로 이미 좋은 가중치에 와 있으므로 **크게 흔들면 배운 것을 잃습니다.** 9주차 파인튜닝에서 말한 것과 **같은 이유**입니다.
> **관찰 포인트**: `DataCollatorWithPadding` 은 **배치 안에서 가장 긴 문장에만** 맞춰 패딩합니다. 전체를 64로 고정하는 것보다 **훨씬 빠릅니다.** (동적 패딩)

## 1-3. 결과 확인

**셀 4** — 최종 성적

In [ ]:
m = trainer.evaluate()
for k, v in m.items():
    if isinstance(v, float): print(f"{k:28s} {v:.4f}")

## 2. 실습 6 — 평가 · 오분류 문장 읽기

**셀 5** — 혼동행렬 (6주차 지표 ★)

In [ ]:
pred = trainer.predict(test_tok)
y_pred = pred.predictions.argmax(-1)
y_true = np.array(test_tok["label"])

cm = np.zeros((2, 2), dtype=int)
for t, p in zip(y_true, y_pred): cm[t, p] += 1
print("        예측:부정  예측:긍정")
print(f"실제:부정  {cm[0,0]:6d}  {cm[0,1]:8d}")
print(f"실제:긍정  {cm[1,0]:6d}  {cm[1,1]:8d}")

**셀 6** — 틀린 문장을 직접 읽는다 ★★

In [ ]:
import torch.nn.functional as F
probs = F.softmax(torch.tensor(pred.predictions), dim=-1).numpy()
wrong = np.where(y_pred != y_true)[0]
conf  = probs[wrong].max(-1)
order = wrong[np.argsort(-conf)]                    # 확신하고 틀린 것부터

print(f"오분류 {len(wrong)}건 / {len(y_true)}건\n")
for i in order[:6]:
    print(f"[정답 {y_true[i]} → 예측 {y_pred[i]} | 확신 {probs[i].max():.2f}]")
    print(f"  {test_tok['document'][i]}\n")

> **관찰 포인트 ★★**: 틀린 문장들을 **소리 내어 읽게 하세요.** 대개 이런 것들입니다 — ① **반어·비꼼** (*"연기 참 잘하시네요 ^^"*) ② **양가적 평가** (*"영상은 좋은데 스토리가 별로"*) ③ **레이블 자체가 애매**하거나 잘못된 것 (*"ㅋㅋㅋ"*) ④ 너무 짧아 정보가 없는 것
> **핵심 ★**: *"확신하고 틀린 것"* 부터 보는 것이 오류 분석의 기본입니다. 그중 **데이터 레이블 자체가 이상한 것**을 찾으면, *"정확도 100% 가 불가능한 이유"* 를 스스로 알게 됩니다. **미니 프로젝트 보고서 ⑤(해석과 한계)의 연습**이기도 합니다.

## 3. 실습 7 — 저장 → 새 커널에서 재사용

**셀 7** — 저장 ★ 토크나이저도 함께

In [ ]:
SAVE = "models/nsmc-klue-bert"
trainer.save_model(SAVE)          # config.json + model.safetensors
tok.save_pretrained(SAVE)         # ★★ 반드시 함께 저장

import os
print(os.listdir(SAVE))

> **핵심 ★★ (기말 출제 지점)**: **토크나이저를 함께 저장해야 하는 이유** — 모델은 **숫자 인덱스**를 학습했습니다. 다른 토크나이저를 쓰면 **같은 단어가 다른 번호**가 되어 모델이 전혀 다른 입력을 받습니다. **10주차에 "어휘사전을 모델과 함께 저장하라"** 고 한 것과 **완전히 같은 이유**입니다.

**셀 8** — ★ 커널을 재시작한 뒤 이 셀부터 실행

In [ ]:
# Jupyter 메뉴: Kernel → Restart Kernel   (진짜 재사용인지 확인하기 위해)
import torch
from transformers import pipeline

clf = pipeline("text-classification", model="models/nsmc-klue-bert",
               tokenizer="models/nsmc-klue-bert",
               device=0 if torch.cuda.is_available() else -1)

my_texts = [
    "이건 진짜 인생영화다",
    "돈 아까움 시간 낭비",
    "배우는 좋았는데 감독이 아쉽네",
    "그냥 볼 만은 했어요",
    "졸다가 나왔다",
]
for t in my_texts:
    r = clf(t)[0]
    label = "긍정" if r["label"].endswith("1") else "부정"
    print(f"{t:22s} → {label} ({r['score']:.3f})")

> **핵심 ★**: **커널을 재시작하고도 동작하면 "재사용 가능한 모델"** 입니다. 이것이 이력서에 쓸 수 있는 형태입니다 — *"한국어 영화 리뷰 감성 분류 모델을 파인튜닝하고 저장·배포했다."*

**셀 9** — 결과 요약 파일로 남긴다

In [ ]:
report = f"""# NSMC 감성 분류 결과
- 모델: klue/bert-base (파인튜닝)
- 데이터: NSMC 서브셋 train 20,000 / test 4,000, max_length 64, 1 epoch
- 정확도: {m['eval_accuracy']:.4f} / F1: {m['eval_f1']:.4f}
- 10주차 LSTM 정확도: (여기에 본인 값 기입)
- 오분류 유형: 반어 / 양가적 평가 / 레이블 애매
"""
open("results/nsmc_report.md", "w", encoding="utf-8").write(report)
print(report)